# 02 — Access indicators and the shortlist

Starting point: `data/processed/analysis_dataset.csv` — 3,180 localities, population figures
and 33 columns of health units.

Two indicators, on two axes:
- **care** — does the locality have a family doctor?
- **medication** — does the locality have a pharmacy?

Combined, they give four situations, and those are the answer to the business question.

The indicators are now also produced by `src/build_dataset.py`; this notebook is where they
were designed and is kept as the record of that reasoning.

## 1. Load the data

One thing to remember: `dtype={"siruta_code": "str"}` on read. Without it the code becomes a
number and later joins break.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/analysis_dataset.csv", dtype={"siruta_code": "str"})
df.shape

(3180, 47)

## 2. Look at what you have

Search short when filtering column names. `"armaci"` would match only `Farmacii` —
`Puncte farmaceutice` contains `farmac-e-utice`, not `farmac-i-i`. A filter that is too strict
raises nothing; it just returns less, and you believe you have seen everything.

In [2]:
[c for c in df.columns if "armac" in c]

['Depozite farmaceutice', 'Farmacii', 'Puncte farmaceutice', 'has_pharmacy']

## 3. First indicator: `has_family_doctor`

A `True` / `False` column: the locality has at least one family-medicine practice.

Comparing a column with `> 0` produces `True`/`False` for every row at once, and `.sum()` on a
boolean column counts the `True` values, because `True` equals 1.

In [3]:
df["has_family_doctor"] = df["Cabinete medicale de familie"] > 0
df["has_family_doctor"].sum()

np.int64(2821)

## 4. Second indicator: `has_pharmacy`

This is a decision, not just a line of code.

There are two columns: `Farmacii` and `Puncte farmaceutice`. A pharmacy point is the reduced
form, permitted precisely in localities too small to sustain a full pharmacy.

- `Farmacii` only → 2,037 localities
- `Farmacii` **or** `Puncte farmaceutice` → 2,553 localities

516 localities hang on that choice. They are counted here: excluding them would declare those
localities cut off from medication when residents do have somewhere to fill a prescription.
The caveat, recorded in the README: a pharmacy point holds less stock and keeps shorter hours,
so "has access" does not mean equal access.

Each condition needs its own parentheses. `|` binds tighter than `>`, so without them Python
reads `a > (0 | b) > 0` and raises.

In [4]:
df["has_pharmacy"] = (df["Farmacii"] > 0) | (df["Puncte farmaceutice"] > 0)
df["has_pharmacy"].sum()

np.int64(2553)

## 5. The four situations

`pd.crosstab` counts how many rows fall into each combination.

Look at the `False` / `False` corner — the localities with neither.

In [5]:
pd.crosstab(df["has_family_doctor"], df["has_pharmacy"])

has_pharmacy,False,True
has_family_doctor,,
False,169,190
True,458,2363


## 6. Who they are

Average 65+ share for each of the four groups.

In [6]:
df.groupby(["has_pharmacy", "has_family_doctor"])["share_65plus"].mean()

has_pharmacy  has_family_doctor
False         False                0.236058
              True                 0.211674
True          False                0.193260
              True                 0.187851
Name: share_65plus, dtype: float64

The gradient is monotone: the fewer services a locality has, the older it is.

This is correlation, not causation, and the direction cannot be established from this data.
A commune ages and loses its practice for lack of patients, or the services disappear and
young families leave — almost certainly both, reinforcing each other.

## 7. The localities with neither

`~` negates a boolean column. Each condition needs its own parentheses, the same rule as `|`.

Filtering and ranking are separate operations on different objects: the mask goes inside
`df[...]` to produce a filtered DataFrame, and only then does `nlargest` apply.

In [7]:
mask = (~df["has_family_doctor"]) & (~df["has_pharmacy"])
without_access = df[mask]

print(f"{len(without_access)} localities, "
      f"{without_access['population_total'].sum():,} people, "
      f"{without_access['population_65plus'].sum():,} of them aged 65+")

without_access.nlargest(15, "share_65plus")[
    ["county", "locality", "population_total", "population_65plus", "share_65plus"]
]

169 localities, 224,756 people, 47,750 of them aged 65+


,county,locality,population_total,population_65plus,share_65plus
1588,Hunedoara,BATRANA,109,51,0.467890
744,Buzau,PARDOSI,287,128,0.445993
1599,Hunedoara,CERBAL,388,171,0.440722
1597,Hunedoara,BUNILA,309,128,0.414239
854,Caras-Severin,BREBU NOU,328,135,0.411585
1624,Hunedoara,TOMESTI,1001,394,0.393606
2781,Teleorman,UDA-CLOCOCIOV,1149,441,0.383812
714,Buzau,CHILIILE,397,151,0.380353
742,Buzau,ODAILE,596,218,0.365772
1608,Hunedoara,LELESE,327,119,0.363914


## 8. The shortlist: ten localities

Two rules turn 169 candidates into 10, each chosen explicitly and defended in the README.

**Minimum 1,000 inhabitants** (113 candidates left). Picked at the knee of the trade-off curve:
with no threshold the ten centres would serve 1,936 elderly people, about 190 each — which is
exactly why the doctor left in the first place. At a 2,000 threshold the entry bar falls below
the national average of 18.7%, meaning the list starts selecting large communes that happen to
lack services rather than ageing ones.

**At most 2 localities per county.** Without it Teleorman takes 5 of the 10 places. The
constraint is nearly free: elderly people served go from 4,762 to 4,798 and counties covered
from 5 to 7, at a cost of 0.8 percentage points on the entry bar.

Order matters. `sort_values` rather than `nlargest`, because `nlargest(10)` would discard the
candidates needed to replace the ones the per-county cap removes. `groupby(...).head(2)` takes
the first two rows of each group *in the current order*, so the sort has to come first, and
the cut to ten has to come last.

In [8]:
candidates = df[
    (~df["has_family_doctor"])
    & (~df["has_pharmacy"])
    & (df["population_total"] >= 1000)
]
print(f"{len(candidates)} candidates above the threshold")

ranked = candidates.sort_values("share_65plus", ascending=False)
capped = ranked.groupby("county").head(2)
shortlist = capped.head(10)

shortlist[["county", "locality", "population_total", "population_65plus", "share_65plus"]]

113 candidates above the threshold


,county,locality,population_total,population_65plus,share_65plus
1624,Hunedoara,TOMESTI,1001,394,0.393606
2781,Teleorman,UDA-CLOCOCIOV,1149,441,0.383812
2777,Teleorman,FANTANELE,1248,426,0.341346
2990,Valcea,PIETRARI,2891,971,0.335870
1956,Mehedinti,OBARSIA DE CAMP,1502,458,0.304927
886,Caras-Severin,NAIDAS,1077,322,0.298979
1611,Hunedoara,MARTINESTI,1008,297,0.294643
3004,Valcea,STROESTI,2532,743,0.293444
1426,Giurgiu,ISVOARELE,1334,384,0.287856
965,Cluj,MANASTIRENI,1263,362,0.286619


In [9]:
print(f"elderly people served : {shortlist['population_65plus'].sum():,}")
print(f"counties covered      : {shortlist['county'].nunique()}")
print(f"lowest 65+ share      : {shortlist['share_65plus'].min():.1%}")

elderly people served : 4,798
counties covered      : 7
lowest 65+ share      : 28.7%
